# T54-产业高频跟踪

## 项目简介
本项目用于跟踪和分析各行业的高频指标，通过小波分析计算行业景气度分位点和趋势方向。

---
## 第一章：环境配置与依赖导入

In [ ]:
# 第一章：环境配置与依赖导入
import pandas as pd
import numpy as np
import sqlalchemy
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import plotly.graph_objects as go
import pywt
import scipy.stats as stats
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# 导入配置
from config import DB_BOND_URL, DB_STOCK_URL, DB_YQ_URL, PG_URL
from src import DataProcessor, WaveletAnalyzer, IndustryTracker

print("依赖导入完成！")

---
## 第二章：数据库连接与数据准备

In [ ]:
# 第二章：数据库连接与数据准备
# 创建数据库连接
sql_engine_bond = sqlalchemy.create_engine(DB_BOND_URL, poolclass=sqlalchemy.pool.NullPool)
sql_engine_stock = sqlalchemy.create_engine(DB_STOCK_URL, poolclass=sqlalchemy.pool.NullPool)
sql_engine_yq = sqlalchemy.create_engine(DB_YQ_URL, poolclass=sqlalchemy.pool.NullPool)

cursor_bond = sql_engine_bond.connect()
cursor_stock = sql_engine_stock.connect()
cursor_yq = sql_engine_yq.connect()

print("数据库连接成功！")

# 获取产业跟踪指标信息
basic_info = pd.read_sql("SELECT * FROM 产业跟踪指标信息", cursor_yq)
print(f"共有 {len(basic_info)} 个行业配置")
display(basic_info.head())

---
## 第三章：核心算法实现 - 小波分析

In [ ]:
# 第三章：核心算法实现 - 小波分析
def handle_outliers(df):
    """处理异常值"""
    if df.empty:
        return df
    Q1 = df['CLOSE'].quantile(0.25)
    Q3 = df['CLOSE'].quantile(0.75)
    IQR = Q3 - Q1
    outliers_indices = df[
        (df['CLOSE'] < (Q1 - 1.5 * IQR)) | 
        (df['CLOSE'] > (Q3 + 1.5 * IQR))
    ].index
    df.loc[outliers_indices, 'CLOSE'] = np.nan
    df['CLOSE'] = df['CLOSE'].interpolate(method='linear')
    df.ffill(inplace=True)
    df.bfill(inplace=True)
    return df[['DT', 'CLOSE']]

def get_value_cycle(df, period_cycle, period_trend, level):
    """使用小波分析计算周期分位点和趋势"""
    df['DT'] = pd.to_datetime(df['DT'])
    values = df['CLOSE'].values
    wavelet = pywt.Wavelet('db2')
    coeffs = pywt.wavedec(values, wavelet)
    
    level = int(level)
    period_cycle = int(period_cycle)
    period_trend = int(period_trend)
    
    recon = pywt.waverec(
        coeffs[:level+1] + [np.zeros_like(c) for c in coeffs[level+1:]], 
        wavelet
    )
    
    if len(recon) < abs(period_cycle):
        recent_cycle = recon
    else:
        recent_cycle = recon[-period_cycle:]
    
    latest_value = recon[-1]
    percentile = stats.percentileofscore(recent_cycle, latest_value)
    
    if len(recon) < abs(period_trend):
        trend_value = recon[-len(recon) + 1] if len(recon) > 1 else recon[-1]
    else:
        trend_value = recon[-period_trend]
    
    trend_diff = latest_value - trend_value
    trend = 1 if trend_diff > 0 else (-1 if trend_diff < 0 else 0)
    
    return percentile, trend

print("核心算法定义完成！")

---
## 第四章：数据处理函数

In [ ]:
# 第四章：数据处理函数
def get_min_date(trade_codes, cursor):
    """获取多个指标的最小日期"""
    dates = []
    for code in trade_codes:
        result = pd.read_sql(
            f"SELECT max(DT) as max_dt FROM edb.edbdata WHERE TRADE_CODE = '{code}'", 
            cursor
        )
        if not result.empty and result['max_dt'].iloc[0] is not None:
            dates.append(result['max_dt'].iloc[0])
    dates.append(datetime.now().date())
    return min(dates) if dates else datetime.now().date()

def calculate_yoy_12ma(trade_code, date_end, cursor):
    """转12月移动平均年同比"""
    df = pd.read_sql(
        f"SELECT DT, CLOSE FROM edb.edbdata WHERE TRADE_CODE = '{trade_code}' AND DT <= '{date_end}'",
        cursor
    )
    df = handle_outliers(df)
    
    if df.empty:
        return pd.DataFrame(columns=['DT', 'CLOSE']), pd.DataFrame(columns=['DT', 'CLOSE'])
    
    df['DT'] = pd.to_datetime(df['DT'])
    df.set_index('DT', inplace=True)
    df = df.resample('ME').last()
    df1 = df.copy()
    
    df['CLOSE'] = df['CLOSE'].pct_change(12, fill_method=None) * 100
    df1['CLOSE'] = df1['CLOSE'].pct_change(12, fill_method=None) * 100
    
    date_range = pd.date_range(start=df.index.min(), end=df.index.max())
    df = df.reindex(date_range)
    df1 = df1.reindex(date_range)
    
    df['CLOSE'] = df['CLOSE'].interpolate(method='linear')
    df1['CLOSE'] = df1['CLOSE'].interpolate(method='linear')
    df.ffill(inplace=True)
    df1.ffill(inplace=True)
    
    df.reset_index(inplace=True)
    df1.reset_index(inplace=True)
    df.rename(columns={'index': 'DT'}, inplace=True)
    df1.rename(columns={'index': 'DT'}, inplace=True)
    df.dropna(inplace=True)
    df1.dropna(inplace=True)
    
    return df[['DT', 'CLOSE']], df1[['DT', 'CLOSE']]

def monthly_yoy_12ma(trade_code, date_end, cursor):
    """月同比12个月移动平均"""
    indicator_x = pd.read_sql(
        f"SELECT DT, CLOSE FROM edb.edbdata WHERE TRADE_CODE = '{trade_code}' AND DT <= '{date_end}' ORDER BY DT",
        cursor
    )
    indicator_x = handle_outliers(indicator_x)
    
    if indicator_x.empty:
        return pd.DataFrame(columns=['DT', 'CLOSE']), pd.DataFrame(columns=['DT', 'CLOSE'])
    
    indicator_x['DT'] = pd.to_datetime(indicator_x['DT'])
    indicator_x.set_index('DT', inplace=True)
    indicator_x = indicator_x.resample('ME').mean()
    indicator_x1 = indicator_x.copy()
    
    date_range = pd.date_range(start=indicator_x.index.min(), end=indicator_x.index.max())
    indicator_x = indicator_x.reindex(date_range)
    indicator_x1 = indicator_x1.reindex(date_range)
    
    indicator_x['CLOSE'] = indicator_x['CLOSE'].interpolate(method='linear')
    indicator_x1['CLOSE'] = indicator_x1['CLOSE'].interpolate(method='linear')
    indicator_x.ffill(inplace=True)
    indicator_x1.ffill(inplace=True)
    
    indicator_x.reset_index(inplace=True)
    indicator_x1.reset_index(inplace=True)
    indicator_x.rename(columns={'index': 'DT'}, inplace=True)
    indicator_x1.rename(columns={'index': 'DT'}, inplace=True)
    indicator_x.dropna(inplace=True)
    indicator_x1.dropna(inplace=True)
    
    return indicator_x[['DT', 'CLOSE']], indicator_x1[['DT', 'CLOSE']]

def highfreq_price_yoy(trade_code, date_end, cursor):
    """高频价格转12月移动平均月均价年同比"""
    indicator_x = pd.read_sql(
        f"SELECT DT, CLOSE FROM edb.edbdata WHERE TRADE_CODE = '{trade_code}' AND DT <= '{date_end}' ORDER BY DT",
        cursor
    )
    
    if indicator_x.empty:
        return pd.DataFrame(columns=['DT', 'CLOSE']), pd.DataFrame(columns=['DT', 'CLOSE'])
    
    indicator_x['DT'] = pd.to_datetime(indicator_x['DT'])
    indicator_x.set_index('DT', inplace=True)
    indicator_x = indicator_x.resample('ME').mean()
    indicator_x1 = indicator_x.copy()
    
    indicator_x['CLOSE'] = indicator_x['CLOSE'].pct_change(12, fill_method=None) * 100
    indicator_x1['CLOSE'] = indicator_x1['CLOSE'].pct_change(12, fill_method=None) * 100
    
    date_range = pd.date_range(start=indicator_x.index.min(), end=indicator_x.index.max())
    indicator_x = indicator_x.reindex(date_range)
    indicator_x1 = indicator_x1.reindex(date_range)
    
    indicator_x['CLOSE'] = indicator_x['CLOSE'].interpolate(method='linear')
    indicator_x1['CLOSE'] = indicator_x1['CLOSE'].interpolate(method='linear')
    indicator_x.ffill(inplace=True)
    indicator_x1.ffill(inplace=True)
    
    indicator_x.reset_index(inplace=True)
    indicator_x1.reset_index(inplace=True)
    indicator_x.rename(columns={'index': 'DT'}, inplace=True)
    indicator_x1.rename(columns={'index': 'DT'}, inplace=True)
    
    indicator_x = handle_outliers(indicator_x)
    indicator_x1 = handle_outliers(indicator_x1)
    indicator_x.dropna(inplace=True)
    indicator_x1.dropna(inplace=True)
    
    return indicator_x[['DT', 'CLOSE']], indicator_x1[['DT', 'CLOSE']]

print("数据处理函数定义完成！")

---
## 第五章：行业景气度计算

In [ ]:
# 第五章：行业景气度计算
def calculate_industry_score(industry, basic_p, date_end, cursor):
    """计算单个行业的景气度得分"""
    period_cycle = basic_p[basic_p['行业'] == industry]['周期天数'].iloc[0]
    period_trend = basic_p[basic_p['行业'] == industry]['趋势天数'].iloc[0]
    level = basic_p[basic_p['行业'] == industry]['level'].iloc[0]
    trade_code = basic_p[basic_p['行业'] == industry]['trade_code'].iloc[0]
    name = basic_p[basic_p['行业'] == industry]['名称'].iloc[0]
    
    df = pd.read_sql(
        f"SELECT DT, CLOSE FROM edb.edbdata WHERE TRADE_CODE = '{trade_code}' AND DT <= '{date_end}' ORDER BY DT",
        cursor
    )
    
    if df.empty:
        return pd.DataFrame(columns=['行业', '指标名称', 'DT', '分位点', '趋势'])
    
    df = df.sort_values('DT')
    percentile, trend = get_value_cycle(df, period_cycle, period_trend, level)
    
    result = pd.DataFrame(columns=['行业', '指标名称', 'DT', '分位点', '趋势'])
    result.loc[0] = [industry, name, df['DT'].iloc[-1], percentile, trend]
    
    return result

def calculate_all_industries(basic_p, date_end, cursor):
    """计算所有行业的景气度"""
    final_df = pd.DataFrame()
    
    industries = basic_p['行业'].unique().tolist()
    
    for industry in industries:
        try:
            result = calculate_industry_score(industry, basic_p, date_end, cursor)
            if not result.empty:
                final_df = pd.concat([final_df, result], ignore_index=True)
                print(f"{industry}: 分位点={result['分位点'].iloc[0]:.2f}%, 趋势={result['趋势'].iloc[0]}")
        except Exception as e:
            print(f"{industry}: 计算失败 - {str(e)}")
    
    return final_df

# 计算示例
date_end = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
print(f"计算日期: {date_end}")

# 运行计算
industry_scores = calculate_all_industries(basic_info, date_end, cursor_stock)
print(f"\n共计算 {len(industry_scores)} 个行业")
display(industry_scores.head(10))

---
## 第六章：可视化分析

In [ ]:
# 第六章：可视化分析
def plot_industry_scores(df):
    """绘制行业景气度分布图"""
    import math
    
    df = df.copy()
    df['x'] = np.arcsin(2 * df['分位点'] / 100 - 1)
    df['x'] = df.apply(lambda row: np.pi - row['x'] if row['趋势'] == -1 else row['x'], axis=1)
    df['x'] = df.apply(lambda row: np.pi * 2 + row['x'] if row['x'] < 0 else row['x'], axis=1)
    df['y'] = df['分位点'] / 100
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df['x'],
        y=df['y'],
        mode='markers+text',
        text=df['行业'],
        textposition='top center',
        marker=dict(
            size=15,
            color=df['分位点'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title='分位点')
        ),
        hovertemplate='<b>%{text}</b><br>分位点: %{marker.color:.1f}%<extra></extra>'
    ))
    
    fig.update_layout(
        title='行业景气度分布图',
        xaxis_title='角度',
        yaxis_title='分位点',
        width=1000,
        height=600
    )
    
    fig.show()

# 绘制图表
if not industry_scores.empty:
    plot_industry_scores(industry_scores)

---
## 第七章：数据导出与保存

In [ ]:
# 第七章：数据导出与保存
import os

# 保存到数据库
if not industry_scores.empty:
    industry_scores['DT'] = date_end
    industry_scores.to_sql('行业景气度跟踪', cursor_yq, if_exists='append', index=False)
    print(f"数据已保存到数据库，共 {len(industry_scores)} 条记录")

# 保存到CSV
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, f'行业景气度跟踪_{date_end}.csv')
industry_scores.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"数据已保存到: {output_file}")

# 显示最终结果
print("\n=== 最终结果 ===")
display(industry_scores.sort_values('分位点', ascending=False))

# 关闭连接
cursor_bond.close()
cursor_stock.close()
cursor_yq.close()
print("\n数据库连接已关闭")